In [ ]:
# # Install scikit-rf if not already available
# %pip install scikit-rf matplotlib

In [ ]:
import skrf as rf
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import shutil
import re
from datetime import datetime

In [ ]:
# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _run_suffix(path: Path) -> str:
    """Extract run token e.g. '-0001' from a stem, even when followed by '-parameter'.

    Matches the last -NNN... that is followed by either a non-digit suffix or end-of-stem.
      MWS-sweep-02-0001.s2p          → '-0001'
      MWS-sweep-02-0001-parameter.txt → '-0001'
      04232026_145355-0001.s2p        → '-0001'
    """
    m = re.search(r'(-\d+)(?=-\D|$)', path.stem)
    return m.group(1) if m else ''

def _archive_file(src: Path, dest_dir: Path) -> Path:
    mtime = datetime.fromtimestamp(src.stat().st_mtime)
    dest  = dest_dir / f'{mtime.strftime("%m%d%Y_%H%M%S")}{_run_suffix(src)}{src.suffix}'
    shutil.copy2(src, dest)
    return dest

def _find_matching_txt(s2p: Path, txt_files: list) -> Path | None:
    """Find the txt file whose run suffix matches the given s2p file."""
    suffix = _run_suffix(s2p)
    for txt in txt_files:
        if _run_suffix(txt) == suffix:
            return txt
    return None

def _read_pressure(s2p_path: Path) -> float | None:
    """Find the archived .txt with matching run suffix and extract number after '='.

    Archived s2p: '04232026_145355-0001.s2p'
    Archived txt: '04232026_145355-0001.txt'
    File content:  'Pressure=375'
    """
    suffix      = _run_suffix(s2p_path)
    txt_matches = list(s2p_path.parent.glob(f'*{suffix}.txt'))
    if not txt_matches:
        return None
    text = txt_matches[0].read_text(errors='ignore').strip()
    if '=' not in text:
        return None
    m = re.search(r'[+-]?[\d.]+(?:[eE][+-]?\d+)?', text.split('=', 1)[1])
    return float(m.group()) if m else None

In [ ]:
folder   = 'TOUCHSTONE files'
run_name = 'Scooter_Curved_Anode_Sim1'

# name -> source 'TOUCHSTONE files' directory. Add/remove entries to load & compare more.
DATASETS = {
    'lam_12': r'/home/vpodolsky/CST Studio Result Files/Scooter_Curved_Anode_Sim1_field_lam_12/TOUCHSTONE files',
    'lam_10': r'/home/vpodolsky/CST Studio Result Files/Scooter_Curved_Anode_Sim1_field_lam_10/TOUCHSTONE files',
}

datasets      = {}   # name -> {pressure: Network}
dataset_files = {}   # name -> {pressure: archived Path}
dataset_dest  = {}   # name -> dest_dir

for name, src in DATASETS.items():
    touchstone_dir = Path(src)
    project_dir    = touchstone_dir.parents[1]
    dest_dir       = project_dir / 'RESULTS_TO_PROCESS' / f'{folder} {run_name}_{name}'
    dest_dir.mkdir(parents=True, exist_ok=True)
    dataset_dest[name] = dest_dir

    # --- Differential archive — copy only new run suffixes (originals kept) ---
    archived_suffixes = {_run_suffix(f) for f in dest_dir.glob('*.s2p')}
    source_s2p        = sorted(touchstone_dir.glob('*.s2p'))
    source_txt        = sorted(touchstone_dir.glob('*.txt'))
    missing = [f for f in source_s2p if _run_suffix(f) not in archived_suffixes]

    if missing:
        to_archive = []
        for s2p in missing:
            to_archive.append(s2p)
            txt = _find_matching_txt(s2p, source_txt)
            if txt:
                to_archive.append(txt)
        for f in to_archive:
            _archive_file(f, dest_dir)
        print(f'[{name}] archived {len(missing)} new run(s) → {dest_dir}')
    else:
        print(f'[{name}] archive up to date ({len(archived_suffixes)} run(s)) → {dest_dir}')

    # --- Load networks keyed by pressure ---
    by_p, files_by_p = {}, {}
    for f in sorted(dest_dir.glob('*.s2p')):
        p = _read_pressure(f)
        by_p[p]       = rf.Network(str(f))
        files_by_p[p] = f
    datasets[name]      = dict(sorted(by_p.items(), key=lambda kv: (kv[0] is None, kv[0])))
    dataset_files[name] = files_by_p

print()
for name, d in datasets.items():
    print(f'{name}: {len(d)} run(s) at {sorted(p for p in d if p is not None)} Torr')

# S11 → index (0,0), S21 → index (1,0)
params = {'S11': (0, 0), 'S21': (1, 0)}

In [ ]:
# --- Choose which dataset drives the single-dataset plots below ---
active_dataset = 'lam_10'                      # any key of `datasets`

sel             = datasets[active_dataset]     # {pressure: Network}
swept_pressures = list(sel.keys())
networks        = list(sel.values())
s2p_files       = [dataset_files[active_dataset][p] for p in swept_pressures]

baseline_idx = next((i for i, p in enumerate(swept_pressures) if p == 0), 0)
pressures    = [p for p in swept_pressures if p is not None and p != 0]

print(f'Active dataset: {active_dataset}  ({len(networks)} run(s))')
for f, p in zip(s2p_files, swept_pressures):
    tag = f'{p:.4g} Torr' if p is not None else 'pressure unknown'
    print(f'  {f.name}  →  {tag}')
print(f'\nbaseline_idx = {baseline_idx}  (pressure = {swept_pressures[baseline_idx]} Torr)')
print(f'Non-zero pressures: {pressures}')


In [ ]:
networks = [rf.Network(str(f)) for f in s2p_files]
ntwk = networks[0]
print(f'Network: {ntwk.name}')
print(f'Frequency range: {ntwk.f[0]/1e9:.3f} – {ntwk.f[-1]/1e9:.3f} GHz  ({len(ntwk.f)} points)')
print(f'Number of ports: {ntwk.nports}')

In [ ]:


cmap=plt.cm.tab10
colors=[cmap(i) for i,_ in enumerate(networks)]
for idx, ntwk in enumerate(networks):
    freq_ghz = ntwk.f / 1e9

    fig, axes = plt.subplots(1, 2, figsize=(18, 5))
    fig.suptitle(f'{run_name}, S-Parameters — {ntwk.name} at {swept_pressures[idx]} Torr', fontsize=13)

    # --- Magnitude (dB): S11 and S21 overlaid ---
    ax = axes[0]
    for label, (i, j) in params.items():
        mag_db = 20 * np.log10(np.abs(ntwk.s[:, i, j]) + 1e-30)
        ax.plot(freq_ghz, mag_db, label=label)
    ax.set_xlabel('Frequency (GHz)')
    ax.set_ylabel('Magnitude (dB)')
    ax.set_title(f'{run_name} Magnitude')
    ax.legend()
    ax.grid(True, alpha=0.4)

    # --- Phase: S21 ---
    ax = axes[1]
    i, j = params['S21']
    ax.plot(freq_ghz, np.angle(ntwk.s[:, i, j], deg=True), color='tab:orange')
    ax.set_xlim([min(freq_ghz),max(freq_ghz)])
    ax.set_xlabel('Frequency (GHz)')
    ax.set_ylabel('Phase (degrees)')
    ax.set_title(f'{run_name} S21 Phase')
    ax.grid(True, alpha=0.4)

    plt.tight_layout()
    plt.show()

# --- S21 magnitude zoom: all pressures overlaid ---
fig2, ax2 = plt.subplots(figsize=(10, 5))
ax2.set_title(f'{run_name}, S21 Magnitude — all pressures')
i, j = params['S21']
for idx, ntwk in enumerate(networks):
    mag_db = 20 * np.log10(np.abs(ntwk.s[:, i, j]) + 1e-30)
    ax2.plot(ntwk.f / 1e9, mag_db, color=colors[idx], label=f'{swept_pressures[idx]} Torr')
ax2.set_xlabel('Frequency (GHz)')
ax2.set_ylabel('S21 (dB)')
ax2.legend()
ax2.grid(True, alpha=0.4)
# ax2.set_ylim(-1, 1)
ax2.set_xlim([min(freq_ghz),max(freq_ghz)])
plt.tight_layout()
plt.show()

# --- S11 and S21 all pressures on one plot ---
fig3, ax3 = plt.subplots(figsize=(10, 5))
ax3.set_title(f'{run_name}, S11 & S21 Magnitude — all pressures')
for idx, ntwk in enumerate(networks):
    freq = ntwk.f / 1e9
    for label, (i, j) in params.items():
        mag_db = 20 * np.log10(np.abs(ntwk.s[:, i, j]) + 1e-30)
        ls = '-' if label == 'S21' else ':'
        ax3.plot(freq, mag_db, color=colors[idx], linestyle=ls,
                 label=f'{label} {swept_pressures[idx]} Torr')
ax3.set_xlabel('Frequency (GHz)')
ax3.set_ylabel('Magnitude (dB)')
ax3.set_xlim([min(freq_ghz),max(freq_ghz)])
ax3.legend(ncol=2, fontsize=8)
ax3.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# --- S11 and S21 all pressures on one plot ---
fig3, ax3 = plt.subplots(figsize=(10, 5))
ax3.set_title(f'{run_name}, S11 & S21 Magnitude — all pressures')
for idx, ntwk in enumerate(networks):
    freq = ntwk.f / 1e9
    for label, (i, j) in params.items():
        mag_db = 20 * np.log10(np.abs(ntwk.s[:, i, j]) + 1e-30)
        ls = '-' if label == 'S21' else ':'
        ax3.plot(freq, mag_db, color=colors[idx], linestyle=ls,
                 label=f'{label} {swept_pressures[idx]} Torr')
ax3.set_xlabel('Frequency (GHz)')
ax3.set_ylabel('Magnitude (dB)')
# ax3.set_xlim([min(freq_ghz),max(freq_ghz)])
ax3.legend(ncol=2, fontsize=8)
ax3.grid(True, alpha=0.4)


freq_list=[59.552, 59.56,  59.88,  59.888, 61.744, 61.752, 61.776, 64.76,64.768]
for freq in freq_list:
    ax3.axvline(freq, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# --- Phase: S21 ---

fig,axes=plt.subplots(3,1,figsize=(10,18))
i, j = params['S21']
cmap=plt.cm.tab10
colors=[cmap(i) for i,_ in enumerate(networks)]
phase_0=np.unwrap(np.angle(networks[0].s[:, i, j]))*180/np.pi

for ii,ntwk in enumerate(networks):
    ax=axes[0]
    ax.plot(freq_ghz, np.angle(ntwk.s[:, i, j]), color=colors[ii])
    ax.set_xlabel('Frequency (GHz)')
    ax.set_ylabel('Phase (rads)')
    ax.set_title(f'{run_name}, S21 Phase')
    ax.grid(True, alpha=0.4)

    # ax = axes[1]
    # ax.plot(freq_ghz, np.unwrap(np.angle(ntwk.s[:, i, j])), color=colors[ii])
    # ax.set_xlabel('Frequency (GHz)')
    # ax.set_ylabel('Unwrapped Phase (Rad)')
    # ax.set_title(f'{run_name}, S21 Phase')
    # ax.grid(True, alpha=0.4)

    ax = axes[1]
    ax.plot(freq_ghz, np.unwrap(np.angle(ntwk.s[:, i, j]))*180/np.pi, color=colors[ii])
    ax.set_xlabel('Frequency [GHz]')
    ax.set_ylabel('Unwrapped Phase [deg]')
    ax.set_title(f'{run_name}, S21 Phase')
    # ax.set_xlim([59,59.3])
    ax.set_ylim([-2500,0])

    ax = axes[2]
    phase_n=np.unwrap(np.angle(ntwk.s[:, i, j]))*180/np.pi
    ax.plot(freq_ghz, phase_n-phase_0, color=colors[ii])
    ax.set_xlabel('Frequency [GHz]')
    ax.set_ylabel('dif Unwrapped Phase [deg]')
    ax.set_title(f'{run_name}, S21 Phase')
    # ax.set_xlim([59,59.3])
    # ax.set_ylim([-100,0])

    ax.grid(True, alpha=0.4)

In [ ]:
# find frequencies where "dif Unwrapped Phase" (phase - baseline_phase) > 0 for each network
i, j = params['S21']
baseline_phase = np.unwrap(np.angle(networks[baseline_idx].s[:, i, j])) * 180 / np.pi

dif_positive_freqs = {}
for idx, ntwk in enumerate(networks):
    phase = np.unwrap(np.angle(ntwk.s[:, i, j])) * 180 / np.pi
    dif = phase - baseline_phase
    freqs_pos = freq_ghz[dif > 0]
    dif_positive_freqs[ntwk.name] = freqs_pos
    print(f"{ntwk.name}: {len(freqs_pos)} points > 0")
    print(dif_positive_freqs[ntwk.name])
# dif_positive_freqs maps network name -> numpy array of frequencies (GHz) where dif > 0


In [ ]:
def get_analytic_permitivity(p, T):
    """Return (permittivity, refractive_index) of N2 at pressure p [Torr] and temperature T [K]."""
    n_N2_STP = 1.0002976          # N2 refractive index at 760 Torr, 0 °C
    n = 1 + (n_N2_STP - 1) * (p / 760) * (273.15 / T)
    return n**2, n

pressures = swept_pressures
PLOT_PRESSURES = {100, 300, 500}   # non-zero pressures to analyze (set membership)
n_vac = 1

T = 293.15          # K
c = 2.99792458e8    # m/s --speed of light
d = 0.226           # m  — path length

f_hz     = networks[0].f
lam      = c / f_hz          # m -- MWI wavelength
freq_ghz = f_hz / 1e9

i, j            = params['S21']
phase_unwrapped = [np.unwrap(np.angle(ntwk.s[:, i, j])) * 180 / np.pi
                   for ntwk in networks]
baseline_phase  = phase_unwrapped[baseline_idx]

# Keyed by pressure so sim & analytic stay aligned regardless of which pressures are plotted.
dphi_sim      = {}   # pressure -> Δφ_sim(f)      [deg]
dphi_rad_p_T  = {}   # pressure -> Δφ_analytic(f) [deg]
pcolor_idx    = {}   # pressure -> color index (position in `networks`)

# --- Phase difference: simulation vs analytic ---
fig, ax = plt.subplots(figsize=(10, 7))
ax.set_title(rf'{run_name} [{active_dataset}], $\angle S21(p) - \angle S21_{{vacuum}}$')

for idx, (ntwk, phase) in enumerate(zip(networks, phase_unwrapped)):
    p_torr = pressures[idx]
    if idx == baseline_idx or p_torr not in PLOT_PRESSURES:
        continue
    pcolor_idx[p_torr]   = idx
    dphi_sim[p_torr]     = phase - baseline_phase
    ax.plot(freq_ghz, dphi_sim[p_torr],
            label=f'{ntwk.name} ({p_torr} Torr)', color=colors[idx])

    eps, n_N2_P          = get_analytic_permitivity(p_torr, T)
    dphi_rad_p_T[p_torr] = -(n_N2_P - n_vac) * d * 2 * np.pi / lam * 180 / np.pi
    ax.plot(freq_ghz, dphi_rad_p_T[p_torr], linestyle='--', color=colors[idx])

ax.set_xlabel('Frequency (GHz)')
ax.set_ylabel('ΔPhase (degrees)')
ax.legend(loc='lower center', ncol=2)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

# --- Residual: (sim − analytic) / analytic ---
fig2, ax2 = plt.subplots(figsize=(8, 5))
ax2.set_title(rf'{run_name} [{active_dataset}], $(Δ\Phi_{{sim}}-Δ\Phi_{{analytic}})/Δ\Phi_{{analytic}}$')
for p_torr in sorted(dphi_sim):
    resid = (dphi_sim[p_torr] - dphi_rad_p_T[p_torr]) / dphi_rad_p_T[p_torr] * 100
    ax2.plot(freq_ghz, resid, label=f'{p_torr} Torr', color=colors[pcolor_idx[p_torr]])
ax2.set_xlabel('Frequency (GHz)')
ax2.set_ylabel(' Percent Error (%)')
ax2.set_ylim([-135, 113])
ax2.legend(loc='lower center', ncol=2)
ax2.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
freq_to_plot = np.linspace(59.5, 60, 9)  # GHz
plot_colors = plt.cm.tab10(np.linspace(0, 1, len(freq_to_plot)))

plot_pressures = sorted(dphi_sim)   # pressures actually analyzed (keys of dphi_sim)

fig, ax = plt.subplots(figsize=(8, 5))
ax.set_title(f'{run_name}, ΔPhase vs Pressure')
print(len(freq_ghz))
for color, f_target in zip(plot_colors, freq_to_plot):
    freq_idx = np.argmin(np.abs(freq_ghz - f_target))
    dphi_at_f   = [dphi_sim[p][freq_idx]     for p in plot_pressures]
    dphi_a_at_f = [dphi_rad_p_T[p][freq_idx] for p in plot_pressures]
    ax.plot(plot_pressures, dphi_at_f, color=color, marker='o',
            label=f'{freq_ghz[freq_idx]:.1f} GHz')
    ax.plot(plot_pressures, dphi_a_at_f, color=color, linestyle='--')

ax.set_xlabel('Pressure (Torr)')
ax.set_ylabel('ΔPhase (degrees)')
ax.legend(title='Frequency')
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
# ===========================================================================
# Compare two datasets  (color = pressure, per-dataset colormap + line style)
# ===========================================================================
i, j = params['S21']

num_name = 'lam_12'    # numerator   → plasma,  solid
den_name = 'lam_10'    # denominator → viridis, dashed
num, den = datasets[num_name], datasets[den_name]

if 0.0 not in num or 0.0 not in den:
    raise ValueError(f'Both datasets need a 0 Torr baseline for ΔPhase; '
                     f'have {num_name}:{0.0 in num}, {den_name}:{0.0 in den}')

freq_ghz = num[0.0].f / 1e9
common_p = sorted(p for p in (set(num) & set(den)) if p not in (None, 0.0))
print(f'Common non-zero pressures ({num_name} ∩ {den_name}): {common_p} Torr')

# Per-dataset colormap + line style; color indexes the common pressures.
dcmap  = {num_name: plt.cm.tab10, den_name: plt.cm.tab10}
dstyle = {num_name: '-',           den_name: ':'}
pcolor = {name: {p: cmap(k / max(len(common_p) - 1, 1)) for k, p in enumerate(common_p)}
          for name, cmap in dcmap.items()}

def _legend_dedup(ax, **kw):
    h, l = ax.get_legend_handles_labels()
    seen = dict(zip(l, h))
    ax.legend(seen.values(), seen.keys(), **kw)

# --- S21 magnitude: common pressures only, per-dataset color scheme ---
fig, ax = plt.subplots(figsize=(11, 6))
ax.set_title('S21 Magnitude — dataset comparison')
for name in (num_name, den_name):
    d = datasets[name]
    for p in common_p:
        mag_db = 20 * np.log10(np.abs(d[p].s[:, i, j]) + 1e-30)
        ax.plot(d[p].f / 1e9, mag_db, color=pcolor[name][p], linestyle=dstyle[name],
                label=f'{name} · {p:g} Torr')
ax.set_xlabel('Frequency (GHz)'); ax.set_ylabel('S21 (dB)')
_legend_dedup(ax, ncol=2, fontsize=7); ax.grid(True, alpha=0.4)
plt.tight_layout(); plt.show()

# --- Relative difference in ΔPhase between two datasets, per pressure ---
#     ((Δφ_num − Δφ_den) / Δφ_den) * 100,  with Δφ = ∠S21(p) − ∠S21(0) per dataset
base_num = np.unwrap(np.angle(num[0.0].s[:, i, j]))
base_den = np.unwrap(np.angle(den[0.0].s[:, i, j]))

fig, ax = plt.subplots(figsize=(11, 6))
ax.set_title(rf'$(\Delta\phi_{{{num_name}}} - \Delta\phi_{{{den_name}}}) / \Delta\phi_{{{den_name}}}$ per pressure')
for p in common_p:
    dphi_num = (np.unwrap(np.angle(num[p].s[:, i, j])) - base_num) * 180 / np.pi
    dphi_den = (np.unwrap(np.angle(den[p].s[:, i, j])) - base_den) * 180 / np.pi
    pct = (dphi_num - dphi_den) / dphi_den * 100
    ax.plot(freq_ghz, pct, color=pcolor[den_name][p], label=f'{p:g} Torr')
ax.axhline(0, color='k', linewidth=0.8, linestyle=':')
ax.set_xlabel('Frequency (GHz)'); ax.set_ylabel('Percent difference (%)')
ax.legend(ncol=2, fontsize=8); ax.grid(True, alpha=0.4)
plt.tight_layout(); plt.show()
